# Recommender System — Foundation

**Mục lục**

1. Khái niệm & Problem Formulation
2. User–Item Matrix
3. Feedback Types
4. Task Types
5. Taxonomy — Phân loại hệ thống
6. Các thách thức chung
7. Pipeline tổng quát
8. 3 loại Recommender phổ biến
9. Evaluation Overview

---
## 1. Khái niệm & Problem Formulation

**Recommender System** là hệ thống có nhiệm vụ dự đoán mức độ phù hợp giữa một người dùng (user) và một đối tượng (item), rồi gợi ý những item phù hợp nhất. Dưới góc nhìn toán học, đây là một hàm scoring:

$$f(\text{user},\ \text{item}) \rightarrow \text{score}$$

Hệ thống tính score cho nhiều item, sắp xếp giảm dần, và lấy $K$ item cao nhất — gọi là **Top-K recommendation**.

Có hai góc nhìn chính của bài toán. Góc nhìn thứ nhất là tìm item phù hợp nhất với user, ví dụ như gợi ý phim, nhạc, sản phẩm (Netflix, Shopee). Góc nhìn thứ hai ngược lại là tìm user phù hợp nhất với item, ví dụ như gợi ý quảng cáo hoặc ghép đôi người dùng (Google Ads, Tinder). Dù nhìn theo hướng nào, hai bài toán đều quy về cùng một quy trình: tính score, sắp xếp, lấy Top-K. Do đó, Recommender System thực chất là một bài toán **ranking**.

### Tại sao phải chia thành 2 giai đoạn?

Trong thực tế, tính $f(u, i)$ cho toàn bộ item là không khả thi vì số lượng item có thể lên đến hàng triệu và hệ thống cần phản hồi gần thời gian thực. Do đó phần lớn hệ thống thực tế chia thành **two-stage pipeline**:

**Giai đoạn 1 — Candidate Generation:** lọc nhanh từ toàn bộ item $I$ ra một tập nhỏ ứng viên $C(u) \subset I$, thường vài trăm đến vài nghìn item. Giai đoạn này ưu tiên tốc độ và độ bao phủ (recall), không cần quá chính xác.

**Giai đoạn 2 — Ranking:** chạy model chính xác hơn chỉ trên tập $C(u)$ để xếp hạng và chọn Top-K:

$$\text{TopK}(u) = \underset{i \in C(u)}{\arg\max}\ f(u, i)$$

Sự phân tách này cho phép hệ thống vừa đảm bảo tốc độ (candidate generation nhẹ và nhanh) vừa đảm bảo chất lượng (ranking model phức tạp nhưng chỉ chạy trên tập nhỏ).

---
## 2. User–Item Matrix

Dữ liệu của Recommender System thường được biểu diễn dưới dạng **user–item interaction matrix** $R$, trong đó mỗi hàng là một user, mỗi cột là một item, và mỗi ô $R_{u,i}$ thể hiện mức độ tương tác giữa user $u$ và item $i$.

| user | item1 | item2 | item3 |
|------|-------|-------|-------|
| u1   | 5     | ?     | 3     |
| u2   | ?     | 4     | ?     |
| u3   | 2     | ?     | 5     |

Các ô có giá trị là tương tác đã biết (rating, click, purchase...). Các ô dấu `?` là **missing interactions** — những cặp (user, item) chưa có dữ liệu. Mục tiêu cốt lõi của Recommender System là dự đoán các giá trị còn thiếu này:

$$\hat{R}_{u,i} = f(u, i)$$

### Vấn đề sparsity

Trong thực tế, ma trận $R$ thường **cực kỳ thưa (sparse)** vì số lượng user và item đều rất lớn, trong khi mỗi user chỉ tương tác với một phần rất nhỏ số item. Ví dụ, với 1 triệu user, 100 nghìn item, và mỗi user trung bình xem 50 item, mật độ của ma trận chỉ là:

$$\text{density} = \frac{50}{100{,}000} = 0.05\%$$

Sparsity là nguyên nhân gốc rễ của nhiều thách thức trong Recommender System: khó tính similarity, khó học preference, và dẫn đến cần phải có chiến lược candidate generation thay vì dự đoán toàn bộ ma trận.

---
## 3. Feedback Types

Dữ liệu tương tác trong Recommender System được chia thành hai loại chính dựa trên cách người dùng thể hiện sở thích.

### Explicit Feedback (Phản hồi tường minh)

Explicit feedback là khi người dùng **chủ động** bày tỏ ý kiến về item, ví dụ như đánh giá sao (1–5), like/dislike, hoặc viết review. Loại dữ liệu này có ưu điểm là rõ ràng, dễ diễn giải và trực tiếp phản ánh sở thích của người dùng. Tuy nhiên, nhược điểm lớn là dữ liệu **rất ít** vì người dùng hiếm khi chủ động đánh giá — trên thực tế chỉ một phần nhỏ người dùng để lại rating sau khi xem phim hay mua hàng. Điều này làm cho explicit feedback thường bị sparse hơn so với implicit feedback.

### Implicit Feedback (Phản hồi ngầm định)

Implicit feedback là tín hiệu thu thập được từ **hành vi tự nhiên** của người dùng mà không cần họ chủ động làm gì, bao gồm click, view, thời gian xem (watch time), purchase, add to cart, scroll qua, v.v. Dữ liệu này **phong phú hơn nhiều** nhưng lại **nhiễu hơn** — ví dụ, việc click vào một bài viết không có nghĩa là người dùng thích nó, hay mua một sản phẩm không nhất thiết có nghĩa họ hài lòng. Implicit feedback chỉ cho biết user **đã tương tác**, không cho biết họ **thích hay không**.

### So sánh

| | Explicit Feedback | Implicit Feedback |
|---|---|---|
| Ví dụ | Rating, like, review | Click, view, purchase |
| Lượng dữ liệu | Ít | Nhiều |
| Độ chính xác ý nghĩa | Cao | Thấp (nhiễu) |
| Cần nỗ lực từ user | Có | Không |
| Phổ biến trong thực tế | Ít hơn | Nhiều hơn |

Phần lớn hệ thống thực tế phải làm việc chủ yếu với implicit feedback vì explicit feedback quá hiếm. Điều này đặt ra thách thức đặc thù: làm thế nào để học được sở thích người dùng từ dữ liệu không có nhãn rõ ràng.

---
## 4. Task Types

Recommender System có thể được đặt ra dưới nhiều dạng bài toán khác nhau tùy theo mục tiêu và loại output mong muốn.

### Rating Prediction

Bài toán cổ điển nhất: dự đoán rating mà user sẽ cho item nếu họ tương tác. Output là một con số liên tục, ví dụ rating từ 1 đến 5 sao. Đây là bài toán regression, thường được đánh giá bằng RMSE hoặc MAE. Bài toán này phù hợp khi có explicit feedback nhưng trong thực tế ngày càng ít được dùng vì không phản ánh đúng mục tiêu thực của hệ thống — người dùng không cần biết score chính xác, họ chỉ cần được gợi ý item phù hợp.

### Top-K Recommendation

Bài toán phổ biến nhất hiện nay: với mỗi user, tìm ra $K$ item phù hợp nhất để hiển thị. Output là một danh sách có thứ tự. Đây là bài toán ranking, và cũng là dạng output mà người dùng trực tiếp thấy trên giao diện (homepage Netflix, feed TikTok, v.v.). Metric đánh giá thường là Precision@K, Recall@K, NDCG@K.

### Click-Through Rate (CTR) Prediction

Dự đoán xác suất user sẽ click vào một item nếu nó được hiển thị. Output là một xác suất từ 0 đến 1. Đây là bài toán binary classification, rất phổ biến trong hệ thống quảng cáo và news feed, nơi mà mục tiêu tối ưu là tối đa hóa tỉ lệ tương tác.

### Sequence-aware Recommendation

Gợi ý dựa trên **lịch sử tương tác theo thứ tự thời gian** của user. Ví dụ: user đã xem phim A, B, C — gợi ý phim tiếp theo phù hợp. Bài toán này quan tâm đến ngữ cảnh thời gian và xu hướng thay đổi sở thích theo thời gian, không chỉ đơn thuần là preference tĩnh. Các model phổ biến cho task này bao gồm GRU4Rec, SASRec, BERT4Rec.

### Session-based Recommendation

Tương tự sequence-aware nhưng chỉ dựa trên **session hiện tại** (phiên tương tác ngắn hạn) mà không cần lịch sử dài hạn. Phù hợp với các tình huống anonymous user hoặc khi sở thích ngắn hạn quan trọng hơn lịch sử dài hạn, ví dụ như gợi ý sản phẩm trong một phiên mua sắm dựa trên những gì vừa xem.

### Conversational Recommendation

Hệ thống tương tác với user qua hội thoại để thu thập preference rồi đưa ra gợi ý. Đây là hướng nghiên cứu mới kết hợp giữa Recommender System và Natural Language Processing/Dialogue Systems, trở nên phổ biến hơn với sự phát triển của LLM.

---
## 5. Taxonomy — Phân loại hệ thống

Có nhiều cách phân loại Recommender System tùy theo tiêu chí. Dưới đây là các cách phân loại phổ biến nhất trong nghiên cứu và thực tế.

### Phân loại theo nguồn dữ liệu sử dụng

**Collaborative Filtering (CF)** chỉ dựa vào dữ liệu tương tác user–item, không dùng nội dung item. Ý tưởng là "những user có hành vi giống nhau sẽ thích item giống nhau". CF không cần hiểu item là gì, chỉ cần biết ai đã tương tác với item nào.

**Content-Based Filtering (CB)** dựa vào đặc trưng nội dung của item (thể loại, mô tả, tag...) và profile của user. Ý tưởng là "nếu user thích item A, gợi ý các item có nội dung tương tự A". CB không cần dữ liệu từ user khác.

**Knowledge-Based** dựa vào tri thức miền (domain knowledge) hoặc constraints do user cung cấp trực tiếp. Ví dụ: "Tôi muốn mua laptop dưới 20 triệu, RAM 16GB, dùng cho lập trình". Phương pháp này không phụ thuộc vào lịch sử tương tác, phù hợp với domain có ít dữ liệu hoặc item phức tạp cần nhiều constraint.

**Hybrid** kết hợp hai hoặc nhiều phương pháp trên để bù đắp nhược điểm của nhau.

### Phân loại theo cách học

**Memory-based** (hay neighborhood-based) không train model, mà tính toán trực tiếp dựa trên similarity giữa user hoặc item trong ma trận tương tác. Đơn giản, dễ giải thích nhưng không scale tốt với dữ liệu lớn.

**Model-based** học một model từ dữ liệu training (matrix factorization, neural network...). Sau khi train, model có thể dự đoán nhanh mà không cần tra cứu toàn bộ ma trận, scale tốt hơn.

### Phân loại theo ngữ cảnh

**Context-aware** tích hợp thêm thông tin ngữ cảnh như thời gian, địa điểm, thiết bị, tâm trạng vào quá trình gợi ý. Ví dụ: buổi sáng gợi ý nhạc nhẹ, buổi tối gợi ý nhạc EDM.

**Cross-domain** sử dụng dữ liệu từ nhiều domain khác nhau để cải thiện gợi ý. Ví dụ: dùng lịch sử đọc sách để cải thiện gợi ý phim.

**Social-aware** tích hợp thông tin mạng xã hội (bạn bè, follows, groups) vào mô hình, dựa trên giả định rằng người dùng có xu hướng thích những gì bạn bè họ thích.

### Phân loại theo mô hình toán học

Từ truyền thống đến hiện đại, các mô hình phát triển theo hướng: neighborhood methods → matrix factorization → factorization machines → deep learning models (NeuMF, BERT4Rec, SASRec) → graph-based models (LightGCN) → large language model-based recommenders.

### Sơ đồ tổng quát

```
Recommender System
├── Collaborative Filtering
│   ├── Memory-based (User-based, Item-based)
│   └── Model-based (Matrix Factorization, Deep Learning)
├── Content-Based Filtering
├── Knowledge-Based
└── Hybrid
    ├── Weighted, Switching, Mixed
    ├── Feature Combination, Cascade
    └── Meta-level
```

---
## 6. Các thách thức chung

Recommender System gặp phải nhiều thách thức đặc thù xuất phát từ bản chất của dữ liệu tương tác người dùng–đối tượng, vốn rất lớn, thưa và thay đổi liên tục. Dưới đây là các thách thức quan trọng nhất.

### Data Sparsity (Dữ liệu thưa)

Đây là thách thức cơ bản nhất. Ma trận user–item $R_{u,i}$ trong thực tế thường có mật độ rất thấp (dưới 1%) vì mỗi user chỉ tương tác với một phần nhỏ item. Sparsity làm cho việc tính similarity trở nên không đáng tin cậy (hai user có rất ít item chung để so sánh), khiến các dự đoán dễ bị nhiễu và không ổn định. Nó cũng làm giảm hiệu quả của neighborhood-based methods và gây khó khăn trong việc học biểu diễn (embedding) của các user hoặc item hiếm tương tác.

### Cold Start Problem

Cold start xảy ra khi hệ thống không có đủ dữ liệu để đưa ra gợi ý cho một thực thể mới. Có hai dạng:

**User cold start** xảy ra với người dùng mới chưa có lịch sử tương tác nào. Hệ thống không biết gì về sở thích của họ, nên không thể tìm được user tương tự hay đưa ra gợi ý cá nhân hóa. Đây là trải nghiệm "tờ giấy trắng" — hệ thống phải đoán mò hoặc dựa vào thông tin nhân khẩu học (tuổi, giới tính, địa điểm) nếu có.

**Item cold start** xảy ra khi item mới vừa được thêm vào hệ thống và chưa có ai tương tác. Collaborative Filtering không thể đặt item này vào không gian embedding hay tìm item tương tự. Item mới sẽ không được recommend cho đến khi có đủ interaction, tạo ra vòng lặp bất lợi: không được gợi ý → không có interaction → vẫn không được gợi ý. Content-Based Filtering giải quyết item cold start tốt hơn vì nó dùng feature của item chứ không phụ thuộc vào interaction.

### Scalability

Khi số lượng user ($M$) và item ($N$) tăng lên hàng triệu, chi phí tính toán tăng theo. Với memory-based methods, tính similarity giữa tất cả cặp user có độ phức tạp $O(M^2)$, giữa tất cả cặp item là $O(N^2)$ — hoàn toàn không khả thi ở scale lớn. Hệ thống cần các kỹ thuật như Approximate Nearest Neighbor (ANN), vector indexing (FAISS, ScaNN), hoặc two-stage retrieval để đảm bảo tốc độ phản hồi. Ngoài ra, khi có user/item mới, hệ thống cần cập nhật model theo thời gian thực hoặc gần thời gian thực, điều này tạo ra thách thức về online learning và incremental training.

### Filter Bubble & Popularity Bias

Collaborative Filtering có xu hướng đề xuất các item phổ biến hơn vì chúng có nhiều interaction hơn, dẫn đến hiệu ứng "rich-get-richer". Item phổ biến càng được gợi ý nhiều → càng nhiều người tương tác → càng được gợi ý nhiều hơn. Ngược lại, long-tail items (item ít phổ biến nhưng chiếm phần lớn catalogue) bị bỏ qua dù có thể phù hợp hơn với người dùng cụ thể. Filter bubble là hiện tượng người dùng bị "nhốt" trong vòng lặp nội dung quen thuộc, hệ thống chỉ gợi ý những gì tương tự với những gì họ đã thích, không giúp khám phá nội dung mới, ảnh hưởng tiêu cực đến trải nghiệm dài hạn.

### Gray Sheep & Black Sheep

**Gray sheep** là những user có sở thích không nhất quán với bất kỳ nhóm user nào, khiến việc tìm "người tương tự" trở nên khó khăn. **Black sheep** là những user hoàn toàn khác biệt so với toàn bộ cộng đồng. Với những user này, Collaborative Filtering gần như không thể hoạt động hiệu quả vì nền tảng của nó là tìm similarity.

### Shilling Attacks

Trong hệ thống mở cho phép user đánh giá tự do, có nguy cơ bị thao túng: người bán có thể tạo tài khoản giả để rate cao sản phẩm của mình và rate thấp sản phẩm đối thủ. Nếu không có cơ chế phát hiện, hệ thống sẽ bị nhiễu và đưa ra gợi ý không trung thực.

### Data Quality & Noise

Implicit feedback (click, view) chứa nhiều nhiễu vì không phải mọi tương tác đều phản ánh sở thích thực sự. User có thể click nhầm, xem vô tình, hoặc mua sản phẩm làm quà mà không phải cho bản thân. Việc xử lý và diễn giải implicit feedback là một thách thức không tầm thường.

### Temporal Dynamics (Biến đổi theo thời gian)

Sở thích người dùng thay đổi theo thời gian — phim hot tuần trước có thể không còn hot tuần sau, và sở thích người dùng có thể thay đổi theo mùa, theo tuổi tác, theo xu hướng xã hội. Mô hình tĩnh (train một lần, dùng mãi) sẽ dần lỗi thời. Hệ thống cần có cơ chế cập nhật hoặc mô hình hoá temporal dynamics.

---
## 7. Pipeline tổng quát

Một hệ thống Recommender đầy đủ trong thực tế thường gồm các bước sau, từ dữ liệu thô đến recommendation cuối cùng.

```
[Data Collection]
    ↓
[Feature Engineering]
    ↓
[Candidate Generation]        ← nhanh, recall cao, lọc từ hàng triệu → vài trăm/nghìn item
    ↓
[Ranking Model]               ← chính xác, chạy trên tập nhỏ, tính score chi tiết
    ↓
[Re-ranking & Business Rules] ← diversity, freshness, constraints kinh doanh
    ↓
[Top-K Recommendation]
    ↓
[Logging & Feedback Loop]     ← thu thập user interaction để retrain
```

### Các thành phần chính

**Users** là tập người dùng $U = \{u_1, u_2, ..., u_m\}$. Mỗi user có thể được mô tả bởi thông tin profile (tuổi, giới tính, địa điểm...), lịch sử tương tác, hành vi duyệt, và embedding vector học được.

**Items** là tập đối tượng được gợi ý $I = \{i_1, i_2, ..., i_n\}$. Item có thể là phim, sản phẩm, bài hát, video, bài viết... Mỗi item có metadata (category, tag, genre), nội dung (text, image, audio), và embedding vector.

**Interactions** mô tả hành vi giữa user và item ($R_{u,i}$), có thể là explicit (rating, like) hoặc implicit (click, view, purchase).

**Scoring Model** học hàm $f(u, i) \rightarrow \text{score}$, thể hiện khả năng user thích item — có thể là mức độ quan tâm, xác suất click, hay xác suất mua.

**Re-ranking** là bước cuối sau ranking, điều chỉnh danh sách để đảm bảo diversity (không quá đơn điệu), freshness (ưu tiên item mới), và các ràng buộc kinh doanh như không gợi ý sản phẩm hết hàng hay tăng visibility cho sản phẩm được quảng bá.

**Feedback Loop** là vòng lặp thu thập log từ hành vi người dùng trên recommendation đã hiển thị, dùng để cải thiện model theo thời gian. Đây là điều làm cho hệ thống "học" và tốt lên liên tục.

---
## 8. 3 loại Recommender phổ biến

### 8.1 Collaborative Filtering [(Wikipedia)](https://en.wikipedia.org/wiki/Collaborative_filtering)

Collaborative Filtering (Lọc cộng tác) là phương pháp Recommender System dựa trên **hành vi của người dùng** để đưa ra gợi ý, **không cần biết nội dung của item**.

> **Ý tưởng cốt lõi:** Những người dùng có hành vi giống nhau trong quá khứ sẽ có xu hướng thích những item giống nhau trong tương lai.

Theo nghĩa hẹp hơn và mới hơn, lọc cộng tác là phương pháp đưa ra dự đoán tự động (lọc) về sở thích của người dùng bằng cách sử dụng thông tin sở thích thu thập từ nhiều người dùng (cộng tác). Cách tiếp cận này giả định rằng nếu người A và B có chung quan điểm về một vấn đề, họ có nhiều khả năng đồng ý với nhau về các vấn đề khác, hơn là so với việc ghép ngẫu nhiên A với một người khác. Ví dụ, một hệ thống CF cho chương trình truyền hình có thể dự đoán chương trình nào người dùng có thể thích dựa trên danh sách hạn chế về sở thích của người đó, nhưng sử dụng thông tin thu thập được từ nhiều người dùng. Điều này khác với cách tiếp cận đơn giản hơn là đưa ra điểm trung bình (không cá nhân hóa) cho mỗi item dựa trên số phiếu bầu.

Theo nghĩa tổng quát hơn, lọc cộng tác là quá trình lọc thông tin bằng các kỹ thuật liên quan đến sự hợp tác giữa nhiều tác nhân, quan điểm, nguồn dữ liệu. Các ứng dụng của CF thường liên quan đến các tập dữ liệu rất lớn và đã được áp dụng trên nhiều loại dữ liệu: dữ liệu cảm biến, dữ liệu tài chính, và dữ liệu người dùng từ thương mại điện tử, ứng dụng web.

Động lực thúc đẩy CF xuất phát từ ý tưởng rằng mọi người thường nhận được những đề xuất tốt nhất từ những người có sở thích tương tự. Một vấn đề then chốt là làm thế nào để kết hợp và trọng số hóa sở thích của những người dùng lân cận. Đôi khi người dùng có thể đánh giá ngay lập tức các mặt hàng được đề xuất, tạo ra feedback trực tiếp cho hệ thống.

---

#### Phương pháp luận

Các hệ thống CF có nhiều hình thức, nhưng phần lớn có thể đơn giản hóa thành hai bước: (1) tìm những người dùng có cùng mô hình đánh giá với người dùng hiện tại, và (2) sử dụng xếp hạng từ những người dùng có cùng quan điểm đó để tính toán dự đoán.

Đây là **User-based CF**. Ngoài ra còn có **Item-based CF** (người dùng đã mua X cũng đã mua Y), hoạt động theo cách xây dựng ma trận tương quan giữa các item để xác định mối quan hệ giữa các cặp item, rồi suy luận sở thích của người dùng hiện tại bằng cách đối chiếu với dữ liệu tương tác của họ.

Một hình thức khác của CF dựa trên quan sát ngầm (implicit) về hành vi thông thường thay vì hành vi đánh giá tường minh. Hệ thống quan sát những gì một người dùng đã làm cùng với những gì tất cả người dùng đã làm (họ đã nghe nhạc gì, mua những mặt hàng nào) và dùng dữ liệu đó để dự đoán hành vi tương lai. Ví dụ, sẽ không hữu ích nếu chào bán cho ai đó một album nhạc mà họ đã sở hữu.

---

#### Các loại Collaborative Filtering

Collaborative Filtering được chia thành năm nhóm chính: Memory-based CF, Model-based CF, Hybrid CF, Deep Learning CF, và Context-aware CF.

---

#### 1. Memory-based Collaborative Filtering

Memory-based CF sử dụng trực tiếp **user–item interaction matrix** để đưa ra dự đoán, không cần huấn luyện model. Phương pháp này tính toán trực tiếp từ dữ liệu lịch sử, không học tham số.

> **Ý tưởng:** Nếu hai người dùng có hành vi giống nhau trong quá khứ, họ sẽ thích các item giống nhau trong tương lai.

---

**User-based Collaborative Filtering**

Phương pháp này tìm các user giống với user cần dự đoán, sau đó dùng rating của họ để ước lượng rating của user đó:

$$\hat r_{u,i} = \sum_{v \in N(u)} sim(u,v) \cdot r_{v,i}$$

Trong đó $u$ là user cần dự đoán, $i$ là item cần dự đoán, $N(u)$ là tập user giống với $u$ nhất (neighbors), $sim(u,v)$ là độ tương đồng giữa $u$ và $v$, và $r_{v,i}$ là rating của user $v$ với item $i$.

Để tránh vấn đề scale (user có rating trung bình cao sẽ ảnh hưởng nhiều hơn), thường chuẩn hóa:

$$\hat r_{u,i} = \frac{\sum_{v \in N(u)} sim(u,v) \cdot r_{v,i}}{\sum_{v \in N(u)} |sim(u,v)|}$$

Tử số là tổng weighted rating từ các user tương tự, mẫu số là tổng trọng số để chuẩn hóa về cùng scale.

---

**Item-based Collaborative Filtering**

Thay vì tìm user tương tự, phương pháp này tìm **item tương tự** với item cần dự đoán, dựa trên pattern rating của tất cả user:

$$\hat r_{u,i} = \frac{\sum_{j \in I(u)} sim(i,j) \cdot r_{u,j}}{\sum_{j \in I(u)} |sim(i,j)|}$$

Trong đó $I(u)$ là tập item user $u$ đã rating, $sim(i,j)$ là similarity giữa item $i$ và $j$, và $r_{u,j}$ là rating của user $u$ với item $j$. Phương pháp này thường ổn định hơn User-based CF vì item similarity ít thay đổi hơn user similarity theo thời gian.

---

#### Similarity Measures

**Cosine Similarity** đo góc giữa hai vector rating:

$$sim(u,v) = \frac{\sum_i r_{u,i} r_{v,i}}{\sqrt{\sum_i r_{u,i}^2} \cdot \sqrt{\sum_i r_{v,i}^2}}$$

Tử số là dot product giữa hai vector, mẫu số chuẩn hóa theo độ dài vector. Giá trị nằm trong $[-1, 1]$.

**Pearson Correlation** tính tương quan tuyến tính sau khi trừ đi rating trung bình của mỗi user — điều này giúp loại bỏ bias của user (có những user có thói quen rate cao hoặc rate thấp hơn thực tế so với mặt bằng chung):

$$sim(u,v) = \frac{\sum_i (r_{u,i}-\bar r_u)(r_{v,i}-\bar r_v)}{\sqrt{\sum_i (r_{u,i}-\bar r_u)^2} \cdot \sqrt{\sum_i (r_{v,i}-\bar r_v)^2}}$$

Trong đó $\bar r_u$, $\bar r_v$ là rating trung bình của user $u$ và $v$. Pearson thường tốt hơn Cosine trong bài toán CF vì nó tính đến phong cách đánh giá của từng user.

---

#### 2. Model-based Collaborative Filtering

Model-based CF học một model dự đoán $f(u,i) \rightarrow \text{score}$ từ dữ liệu training thay vì dựa trực tiếp vào similarity. Sau khi train, model có thể inference nhanh.

**Matrix Factorization** là phương pháp nền tảng nhất. Ý tưởng là phân rã ma trận user–item thành hai ma trận latent có kích thước nhỏ hơn:

$$R \approx U V^T$$

Trong đó $U \in \mathbb{R}^{m \times k}$ là user latent matrix, $V \in \mathbb{R}^{n \times k}$ là item latent matrix, và $k \ll \min(m,n)$ là số chiều latent. Mỗi user được biểu diễn bởi vector $U_u \in \mathbb{R}^k$, mỗi item bởi $V_i \in \mathbb{R}^k$. Score được tính bằng dot product:

$$\hat r_{u,i} = U_u \cdot V_i$$

> **Intuition:** Các chiều latent học được có thể tương ứng với các "concept" ẩn, ví dụ như thể loại phim, mức độ nghiêm túc của nội dung... — dù không được đặt nhãn tường minh. Nếu user $u$ thích phim hành động, chiều latent tương ứng trong $U_u$ sẽ có giá trị cao. Item hành động cũng sẽ có giá trị cao ở chiều đó trong $V_i$. Dot product của hai vector sẽ lớn → score cao.

Model được học bằng cách tối thiểu hóa reconstruction error trên các interaction đã biết, có regularization để tránh overfitting:

$$\min_{U,V} \sum_{(u,i) \in R} (r_{u,i} - U_u V_i)^2 + \lambda \left(||U_u||^2 + ||V_i||^2\right)$$

Số hạng đầu tiên tối thiểu hóa sai số dự đoán trên dữ liệu đã biết. Số hạng thứ hai ($\lambda$-regularization) phạt các vector quá lớn, giúp tránh overfitting. Thuật toán tối ưu thường dùng là SGD (Stochastic Gradient Descent) hoặc ALS (Alternating Least Squares).

---

#### 3. Hybrid Collaborative Filtering

Hybrid CF kết hợp memory-based, model-based, và/hoặc content-based để bù đắp nhược điểm của từng phương pháp. Ví dụ, kết hợp CF score và content-based score theo trọng số:

$$\text{score} = \alpha \cdot CF(u,i) + (1-\alpha) \cdot Content(u,i)$$

Mục tiêu của hybrid là giảm cold start, tăng accuracy, và giảm ảnh hưởng của sparsity.

---

#### 4. Deep Learning Collaborative Filtering

Deep learning mở rộng matrix factorization bằng cách thay dot product tuyến tính bằng một neural network phi tuyến:

$$\hat r_{u,i} = f_{NN}(U_u, V_i)$$

Thay vì chỉ dùng $U_u \cdot V_i$, neural network có thể học các tương tác phức tạp và phi tuyến giữa user và item. Ví dụ kiến trúc đơn giản:

```
embedding(user) ──┐
                  concat → dense layers → score
embedding(item) ──┘
```

Các mô hình nổi tiếng bao gồm NeuMF (Neural Matrix Factorization), Wide & Deep, DeepFM, và các mô hình sequence-based như BERT4Rec, SASRec.

---

#### 5. Context-aware Collaborative Filtering

CF truyền thống mô hình hóa tương tác là $R_{u,i}$ — chỉ phụ thuộc vào user và item. Context-aware CF thêm vào chiều ngữ cảnh $c$:

$$R_{u,i,c}$$

Ngữ cảnh $c$ có thể là thời gian (buổi sáng/tối, ngày thường/cuối tuần), địa điểm, thiết bị, tâm trạng, v.v. Ví dụ, cùng một user nhưng buổi sáng thích nhạc nhẹ, buổi tối thích EDM; hoặc khi ở nhà thích phim dài, khi đi làm thích podcast ngắn. Dữ liệu trở thành tensor 3 chiều (hoặc nhiều hơn), và các kỹ thuật như Tensor Factorization hoặc feature-based model được dùng để xử lý.

---

#### Thách thức của Collaborative Filtering

Mặc dù là một trong những phương pháp phổ biến nhất, CF vẫn tồn tại nhiều thách thức khi áp dụng trên dữ liệu lớn trong thực tế. Các vấn đề chủ yếu xuất phát từ đặc điểm của dữ liệu tương tác người dùng–đối tượng, vốn thường rất lớn, thưa thớt và thay đổi liên tục theo thời gian. Những hạn chế này ảnh hưởng trực tiếp đến độ chính xác, khả năng mở rộng và tính ổn định của hệ thống.

---

#### Dữ liệu thưa thớt (Data Sparsity)

Trong các hệ thống recommender thực tế, số lượng người dùng và đối tượng thường rất lớn, trong khi mỗi người dùng chỉ tương tác với một số lượng rất nhỏ các đối tượng. Điều này dẫn đến ma trận user–item $R_{u,i}$ trở nên cực kỳ thưa thớt. Ví dụ, trong một hệ thống có hàng triệu người dùng và hàng trăm nghìn sản phẩm, mỗi người dùng có thể chỉ mua hoặc đánh giá vài chục sản phẩm. Khi đó, phần lớn các phần tử trong ma trận sẽ là giá trị thiếu. Sự thưa thớt này làm cho việc tính toán độ tương đồng giữa người dùng hoặc giữa các đối tượng trở nên khó khăn, bởi vì số lượng phần tử được đánh giá chung là rất ít. Điều này làm giảm độ chính xác của các phương pháp dựa trên similarity, đồng thời làm tăng độ nhiễu trong quá trình dự đoán.

Một hệ quả quan trọng của dữ liệu thưa thớt là hệ thống khó học được sở thích thực sự của người dùng. Khi dữ liệu quá ít, các dự đoán dễ bị thiên lệch hoặc không ổn định. Ngoài ra, dữ liệu sparse cũng làm giảm hiệu quả của các thuật toán neighborhood-based vì việc tìm kiếm người dùng tương tự trở nên khó khăn hơn.

---

#### Cold Start Problem

Vấn đề khởi đầu lạnh xảy ra khi hệ thống không có đủ dữ liệu để đưa ra dự đoán đáng tin cậy. Điều này xảy ra trong hai trường hợp chính: người dùng mới và đối tượng mới.

Trong trường hợp người dùng mới, hệ thống chưa có thông tin về lịch sử tương tác của người dùng đó. Vì Collaborative Filtering dựa trên hành vi trước đây để dự đoán sở thích, nên khi không có dữ liệu, hệ thống không thể xác định người dùng tương tự hoặc các item phù hợp. Do đó, các đề xuất ban đầu thường không chính xác hoặc mang tính ngẫu nhiên. Hệ thống chỉ có thể cải thiện khi người dùng bắt đầu tương tác với các item, ví dụ như rating, click hoặc purchase.

Tương tự, khi một item mới được thêm vào hệ thống, item đó chưa có rating hoặc interaction từ người dùng. Collaborative Filtering không thể xác định item đó giống với item nào khác. Do đó, item mới sẽ không được recommend cho đến khi có đủ người dùng tương tác với nó. Điều này tạo ra độ trễ trong việc khám phá các item mới và làm giảm khả năng đề xuất các sản phẩm mới.

Khác với Collaborative Filtering, các phương pháp content-based có thể giảm vấn đề này vì chúng sử dụng đặc trưng nội dung của item thay vì phụ thuộc hoàn toàn vào interaction.

---

#### Khả năng mở rộng (Scalability)

Khi số lượng người dùng và đối tượng tăng lên, Collaborative Filtering gặp vấn đề nghiêm trọng về khả năng mở rộng. Giả sử có $M$ người dùng và $N$ đối tượng, ma trận tương tác có kích thước $M \times N$. Các thuật toán memory-based thường cần tính similarity giữa các user hoặc item. Nếu tính similarity giữa user, độ phức tạp có thể là $O(M^2)$; nếu tính giữa item là $O(N^2)$. Khi M và N rất lớn (hàng triệu), chi phí tính toán trở nên rất cao. Ngoài ra, nhiều hệ thống recommender cần phản hồi theo thời gian thực, do đó việc tính toán toàn bộ similarity là không khả thi. Điều này buộc hệ thống phải sử dụng các kỹ thuật tối ưu như approximate nearest neighbor, sampling hoặc candidate generation.

Khả năng mở rộng cũng liên quan đến việc cập nhật dữ liệu mới. Khi có user hoặc item mới, hệ thống có thể cần tính toán lại similarity, điều này làm tăng chi phí và độ trễ của hệ thống.

---

#### Synonym Problem (Vấn đề từ đồng nghĩa)

Một vấn đề khác của Collaborative Filtering là các item giống nhau nhưng có tên hoặc biểu diễn khác nhau. Hệ thống không nhận ra rằng chúng có nội dung tương đương và xử lý chúng như các item độc lập. Ví dụ, hai item như "phim thiếu nhi" và "phim dành cho trẻ em" có thể biểu diễn cùng một loại nội dung nhưng được xem là khác nhau trong hệ thống. Điều này làm phân tán dữ liệu tương tác và giảm độ chính xác của recommendation.

Vấn đề này đặc biệt phổ biến trong hệ thống có dữ liệu văn bản hoặc metadata không chuẩn hóa. Một cách tiếp cận để giải quyết là sử dụng topic modeling hoặc embedding để nhóm các item có ý nghĩa tương tự vào cùng một không gian biểu diễn.

---

#### Gray Sheep Problem

Gray sheep là những người dùng có hành vi không nhất quán với bất kỳ nhóm người dùng nào. Những người dùng này có sở thích đa dạng hoặc thay đổi liên tục, khiến việc tìm user tương tự trở nên khó khăn. Collaborative Filtering dựa vào similarity, nên khi user không giống ai, hệ thống không thể đưa ra đề xuất chính xác.

Ngoài ra, còn tồn tại black sheep, là những người dùng có sở thích hoàn toàn khác biệt so với phần còn lại. Trong trường hợp này, việc đề xuất gần như không thể thực hiện hiệu quả. Đây được xem là hạn chế tự nhiên của Collaborative Filtering.

---

#### Shilling Attacks

Trong các hệ thống mở, người dùng có thể tự do đánh giá item. Điều này dẫn đến khả năng người dùng thao túng hệ thống bằng cách rating cao cho sản phẩm của họ và rating thấp cho sản phẩm đối thủ. Các hành vi này được gọi là shilling attacks. Nếu không có cơ chế phát hiện, hệ thống recommendation có thể bị lệch và đưa ra đề xuất sai. Đây là vấn đề quan trọng trong các hệ thống thương mại điện tử hoặc review online.

---

#### Diversity và Long Tail Problem

Collaborative Filtering thường có xu hướng đề xuất các item phổ biến vì chúng có nhiều interaction hơn. Điều này tạo ra hiệu ứng "rich-get-richer", trong đó các item phổ biến ngày càng được đề xuất nhiều hơn. Ngược lại, các item ít phổ biến, dù có thể phù hợp hơn với người dùng, lại ít được đề xuất.

Hiện tượng này làm giảm tính đa dạng của recommendation và hạn chế khả năng khám phá item mới. Ngoài ra, nó cũng ảnh hưởng đến long-tail items, là các item ít phổ biến nhưng chiếm phần lớn catalogue. Nhiều nghiên cứu đã đề xuất các thuật toán để tăng diversity bằng cách đưa thêm các item mới, bất ngờ hoặc ít phổ biến vào danh sách gợi ý.

---
### 8.2 Content-Based Filtering [(Wikipedia)](https://en.wikipedia.org/wiki/Recommender_system#Content-based_filtering)

Content-Based Filtering (Lọc dựa trên nội dung) là phương pháp Recommender System gợi ý item dựa trên **đặc trưng nội dung của item** mà người dùng đã thích trước đó, **không phụ thuộc vào người dùng khác**.

> **Ý tưởng cốt lõi:** Nếu một người dùng thích một item trong quá khứ, họ sẽ có xu hướng thích các item có nội dung tương tự trong tương lai.

Khác với Collaborative Filtering sử dụng hành vi của nhiều người dùng, Content-Based Filtering chỉ sử dụng **profile của chính người dùng** và **feature của item**. Hệ thống học sở thích của người dùng bằng cách phân tích các thuộc tính của các item đã tương tác, sau đó tìm các item có biểu diễn nội dung tương tự. Ví dụ, nếu người dùng thích các bộ phim thuộc thể loại hành động, khoa học viễn tưởng và đạo diễn cụ thể, hệ thống sẽ ưu tiên đề xuất các bộ phim có cùng thể loại hoặc đặc trưng tương tự.

---

#### Phương pháp luận

Content-Based Filtering thường được thực hiện theo các bước sau: (1) biểu diễn item dưới dạng vector đặc trưng nội dung, (2) xây dựng user profile từ các item đã thích, (3) tính độ tương đồng giữa user profile và item, (4) recommend các item có similarity cao nhất.

```
Items → Feature Extraction → User Profile → Similarity → Recommendation
```

---

#### Item Representation

Mỗi item được biểu diễn bằng một vector feature. Các feature có thể đến từ metadata (genre, category, tag), text (TF-IDF, bag-of-words), embedding (Word2Vec, BERT), image feature (CNN), hoặc audio feature.

Ví dụ biểu diễn phim:
```
Inception = [action=1, sci-fi=1, romance=0, Nolan=1]
Titanic   = [action=0, romance=1, drama=1,  Nolan=0]
Batman    = [action=1, dark=1,   Nolan=1,  sci-fi=0]
```

Các vector này tạo thành **item feature matrix** $X_{i,f}$ trong đó $i$ là item và $f$ là feature.

---

#### User Profile Construction

User profile được xây dựng từ các item người dùng đã thích, thường bằng cách lấy trung bình vector feature:

$$u = \frac{1}{|I_u|} \sum_{i \in I_u} x_i$$

Trong đó $I_u$ là tập item user đã tương tác và $x_i$ là vector feature của item $i$. Nói cách khác, **user profile là trung bình của các item đã thích** — đây là một xấp xỉ đơn giản nhưng hiệu quả cho sở thích của user. Có thể làm phức tạp hơn bằng cách weighted average theo rating hoặc thời gian gần đây.

---

#### Similarity Computation & Recommendation

Độ tương đồng giữa user profile và item thường dùng cosine similarity:

$$sim(u, i) = \frac{u \cdot x_i}{||u|| \cdot ||x_i||}$$

Hệ thống chọn Top-K item có similarity cao nhất:

$$\text{TopK} = \underset{i}{\arg\max}\ sim(u, i)$$

---

#### Ưu điểm của Content-Based Filtering

Content-Based Filtering không cần dữ liệu từ người dùng khác nên không bị ảnh hưởng bởi sparsity của user–item matrix, và không có cold start với user mới (miễn là có ít nhất một vài interaction). Gợi ý dễ giải thích vì có thể nói "được gợi ý vì bạn đã thích phim hành động của Nolan" (Explainable AI). Cũng không phụ thuộc vào popularity bias — có thể recommend long-tail items nếu nội dung phù hợp.

#### Nhược điểm của Content-Based Filtering

**Overspecialization** là vấn đề lớn nhất: hệ thống chỉ gợi ý những thứ quá giống với những gì user đã thích, khó khám phá nội dung mới hoặc đa dạng. Chất lượng phụ thuộc nhiều vào **feature engineering** — nếu feature không đủ phong phú hoặc không đại diện tốt cho nội dung, gợi ý sẽ kém. Vẫn có **item cold start** nếu item mới thiếu metadata. Và CB không học được xu hướng cộng đồng — nó không biết rằng item A đang trend hay được nhiều người thích.

---

#### So sánh với Collaborative Filtering

|                 | Content-Based | Collaborative |
|-----------------|---------------|---------------|
| Dựa vào         | nội dung item | hành vi user  |
| Cần user khác   | không         | có            |
| Cold start user | tốt           | kém           |
| Cold start item | kém           | kém           |
| Diversity       | thấp          | cao           |
| Explainable     | dễ            | khó           |
| Popularity bias | thấp          | cao           |

---

#### Ứng dụng Content-Based Filtering

Content-Based Filtering thường được dùng trong news recommendation, movie recommendation, music recommendation, e-commerce product recommendation, và document recommendation. Các platform như Netflix recommend phim cùng thể loại, Spotify recommend nhạc cùng genre, Amazon recommend sản phẩm tương tự, hay Google News recommend bài viết cùng chủ đề — tất cả đều có thành phần content-based trong pipeline của mình.

---
### 8.3 Hybrid Recommender System [(Wikipedia)](https://en.wikipedia.org/wiki/Recommender_system#Hybrid_recommender_system)

Hybrid Recommender System là phương pháp kết hợp **Collaborative Filtering** và **Content-Based Filtering** nhằm tận dụng ưu điểm của cả hai và giảm các nhược điểm riêng lẻ.

> **Ý tưởng:** Kết hợp nhiều phương pháp recommender để cải thiện độ chính xác, giảm cold start và tăng diversity.

Collaborative Filtering học từ hành vi người dùng nhưng gặp khó với cold start và sparsity. Content-Based Filtering học từ nội dung item nhưng dễ bị overspecialization. Hybrid kết hợp cả hai nguồn thông tin để bù đắp cho nhau.

---

#### Phương pháp luận

Hybrid Recommender System có thể được xây dựng theo nhiều cách: Weighted hybrid, Switching hybrid, Mixed hybrid, Feature combination, Cascade hybrid, và Meta-level hybrid.

---

#### 1. Weighted Hybrid

Phương pháp này kết hợp điểm số từ nhiều recommender bằng cách lấy trung bình có trọng số:

$$\text{score}(u,i) = \alpha \cdot CF(u,i) + (1-\alpha) \cdot CB(u,i)$$

Trong đó $CF(u,i)$ là score từ Collaborative Filtering, $CB(u,i)$ là score từ Content-Based, và $\alpha$ là trọng số kết hợp. $\alpha$ lớn ưu tiên CF, $\alpha$ nhỏ ưu tiên Content-Based. Trọng số $\alpha$ có thể được tuning hoặc học tự động.

---

#### 2. Switching Hybrid

Hệ thống chuyển đổi giữa các phương pháp tùy theo điều kiện. Ví dụ: nếu user mới (ít lịch sử) thì dùng Content-Based, nếu user có nhiều history thì dùng CF. Cách này đơn giản, dễ triển khai và giải quyết tốt cold start:

```python
if len(user_history) < threshold:
    recommend = content_based(user)
else:
    recommend = collaborative_filtering(user)
```

---

#### 3. Mixed Hybrid

Hiển thị kết quả từ nhiều recommender cùng lúc. Ví dụ: Top 5 từ CF hiển thị cùng Top 5 từ Content-Based, danh sách recommendation là hợp của cả hai:

```
recommend = CF_topK ∪ CB_topK
```

Cách này tăng diversity nhưng khó kiểm soát thứ tự tổng thể.

---

#### 4. Feature Combination

Kết hợp feature từ nhiều nguồn rồi train một model duy nhất. Ví dụ, concatenate user embedding (từ CF), item embedding (từ CF), và content features (từ CB) rồi đưa qua một neural network hoặc gradient boosting model. Đây là cách tiếp cận phổ biến trong hệ thống production hiện đại:

```
features = [user_embedding, item_embedding, content_features]
score = model(features)  # Logistic Regression, Neural Network, Gradient Boosting...
```

---

#### 5. Cascade Hybrid

Một recommender lọc trước, recommender khác xếp hạng sau. Đây chính là two-stage pipeline: CF (hoặc retrieval model) tạo candidate set, Content-Based (hoặc ranking model phức tạp hơn) re-rank trên tập đó:

```
Candidate Generation (CF/retrieval) → candidate set → Ranking (CB/deep model) → Top-K
```

Cascade giảm chi phí tính toán (ranking model chỉ chạy trên tập nhỏ) và tăng độ chính xác tổng thể.

---

#### 6. Meta-level Hybrid

Một recommender tạo model/profile, recommender khác dùng model đó. Ví dụ: Content-Based tạo user profile dưới dạng vector preference, CF dùng profile đó thay vì interaction matrix. Cách này tích hợp sâu hơn giữa hai phương pháp:

```
Content-Based → user profile → Collaborative Filtering → recommendation
```

---

#### Ưu điểm của Hybrid Recommender

Hybrid có nhiều ưu điểm so với từng phương pháp đơn lẻ: giảm cold start (CB giúp khi user/item mới), tăng accuracy, giảm ảnh hưởng của sparsity, tăng diversity, và ổn định hơn về tổng thể.

#### Nhược điểm

Nhược điểm là phức tạp hơn để xây dựng và maintain, khó tuning các hyperparameter (như $\alpha$), chi phí tính toán cao hơn, và khó triển khai trong môi trường production với yêu cầu latency thấp.

---

#### Ví dụ thực tế

**Netflix** kết hợp collaborative filtering, content-based metadata (thể loại, diễn viên, đạo diễn), và deep learning hybrid model để tối ưu hóa recommendation. **Amazon** dùng item-based collaborative filtering kết hợp product metadata và hybrid ranking. **Spotify** sử dụng collaborative filtering, audio content features trích xuất từ bản nhạc, và hybrid recommendation trong Discover Weekly. Những hệ thống lớn này đều không chỉ dùng một phương pháp đơn lẻ.

#### Khi nào nên dùng Hybrid

Hybrid thường là lựa chọn tốt khi dataset sparse, khi có metadata item, khi số lượng user mới nhiều, khi cần độ chính xác cao, hoặc khi xây dựng production system lớn có đủ engineering capacity để maintain pipeline phức tạp.

---
## 9. Evaluation Overview

Đánh giá Recommender System là một vấn đề không tầm thường, vì "recommendation tốt" có thể nghĩa là nhiều thứ khác nhau tùy theo góc nhìn và mục tiêu kinh doanh. Có ba nhóm phương pháp đánh giá chính.

### Offline Evaluation

Đây là cách đánh giá phổ biến nhất trong nghiên cứu. Dữ liệu interaction được chia thành train/validation/test set, model được train trên train set và đánh giá trên test set. Không cần deploy hệ thống thực tế.

**Rating Prediction Metrics** — dùng khi bài toán là dự đoán rating:

RMSE (Root Mean Square Error) đo sai số trung bình bình phương của dự đoán rating:

$$\text{RMSE} = \sqrt{\frac{1}{|\mathcal{T}|} \sum_{(u,i) \in \mathcal{T}} (r_{u,i} - \hat r_{u,i})^2}$$

MAE (Mean Absolute Error) tương tự nhưng ít bị ảnh hưởng bởi outlier hơn:

$$\text{MAE} = \frac{1}{|\mathcal{T}|} \sum_{(u,i) \in \mathcal{T}} |r_{u,i} - \hat r_{u,i}|$$

**Ranking Metrics** — dùng khi bài toán là Top-K recommendation (phổ biến hơn trong thực tế):

*Precision@K* đo trong K item được gợi ý, bao nhiêu % là relevant (item mà user thực sự thích):

$$\text{Precision@K} = \frac{|\text{relevant items trong Top-K}|}{K}$$

*Recall@K* đo trong tất cả relevant items của user, bao nhiêu % được gợi ý trong Top-K:

$$\text{Recall@K} = \frac{|\text{relevant items trong Top-K}|}{|\text{tổng relevant items của user}|}$$

*NDCG@K (Normalized Discounted Cumulative Gain)* quan tâm đến **thứ tự** trong danh sách — item relevant ở vị trí cao hơn được thưởng nhiều hơn. Đây là metric phổ biến nhất trong RecSys vì nó phản ánh rằng người dùng thường chú ý nhiều hơn vào các vị trí đầu danh sách:

$$\text{DCG@K} = \sum_{k=1}^{K} \frac{rel_k}{\log_2(k+1)}, \qquad \text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

Trong đó $rel_k \in \{0, 1\}$ là nhãn relevance của item ở vị trí $k$, và IDCG là DCG tối ưu (khi xếp tất cả relevant items lên đầu). NDCG = 1 là kết quả hoàn hảo.

*Hit Rate@K (HR@K)* đo tỉ lệ user có ít nhất một relevant item trong Top-K:

$$\text{HR@K} = \frac{|\text{users có ít nhất 1 hit trong Top-K}|}{|\text{tổng users}|}$$

*MRR (Mean Reciprocal Rank)* đo trung bình nghịch đảo hạng của relevant item đầu tiên:

$$\text{MRR} = \frac{1}{|U|} \sum_{u} \frac{1}{\text{rank của relevant item đầu tiên của } u}$$

**Beyond-Accuracy Metrics** — đánh giá các khía cạnh khác ngoài độ chính xác:

*Coverage* đo tỉ lệ item trong catalogue được hệ thống recommend ít nhất một lần — hệ thống có coverage cao ít bị popularity bias hơn.

*Diversity* đo mức độ đa dạng trong danh sách recommendation của mỗi user. Danh sách đa dạng giúp user khám phá nội dung mới và tránh filter bubble.

*Novelty* đo mức độ "mới" của item được gợi ý — item ít phổ biến và ít được user biết đến có novelty cao hơn.

*Serendipity* đo mức độ bất ngờ nhưng thú vị của recommendation — item serendipitous là item mà user không ngờ mình sẽ thích nhưng thực tế lại thích.

---

### Online Evaluation (A/B Testing)

Offline evaluation có giới hạn quan trọng: nó chỉ đánh giá trên dữ liệu đã có, không phản ánh được hành vi thực của user trên hệ thống mới. Một model có NDCG cao hơn trên offline test set không chắc sẽ tốt hơn trong thực tế.

**A/B testing** là cách đánh giá trực tiếp trên người dùng thực: chia user thành hai nhóm, nhóm A dùng hệ thống cũ (control), nhóm B dùng hệ thống mới (treatment), và so sánh các business metric như CTR (Click-Through Rate), conversion rate, watch time, revenue, user retention. Đây là phương pháp đánh giá đáng tin cậy nhất nhưng tốn kém và cần thời gian để thu thập đủ data.

---

### User Studies

Đánh giá thông qua khảo sát hoặc thí nghiệm có người dùng thực tham gia. Thường dùng để đánh giá các khía cạnh khó đo tự động như sự hài lòng, trustworthiness (tin tưởng vào gợi ý), explainability (dễ hiểu lý do gợi ý), hay perceived serendipity. User study tốn nhiều công sức nhưng cung cấp insight sâu về trải nghiệm người dùng.

---

### Lưu ý quan trọng về Evaluation

Offline metrics và online performance có thể không tương quan tốt. Một model tối ưu cho NDCG trên test set không chắc tốt hơn trong A/B test. Nguyên nhân là do **exposure bias**: trong test set, chỉ có dữ liệu của các item đã được hiển thị cho user (selection bias), không đại diện cho toàn bộ preference space. Ngoài ra, một hệ thống tối ưu thuần túy cho accuracy có thể hy sinh diversity và novelty, dẫn đến filter bubble và user churn dài hạn. Do đó, evaluation tốt cần kết hợp nhiều loại metric và không nên chỉ nhìn vào một con số duy nhất.